### Import modules

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout

# Ensure reproducibility
np.random.seed(42)
tf.random.set_seed(42)


2025-12-17 11:59:52.353248: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Load Data

In [2]:
data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/results/pattern_embeddings/gemini_pattern_embedding_v3.csv')
# data = pd.read_csv('/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/experiments-llm/embeddings/Gemini_embeddings.csv')

In [3]:
data = data.drop(columns=['file'])
data

,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10,...,dim_760,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767,dim_768,pattern
0,0.008586,-0.000224,-0.073952,0.040551,0.041076,0.051949,0.053746,0.020028,0.009389,0.006378,...,0.031212,-0.029720,0.062391,0.007221,0.005423,-0.022622,-0.054182,0.068976,-0.041483,Advanced LLM Prompting
1,0.025723,0.010738,-0.067647,0.041301,0.041939,0.043754,0.024987,0.018802,0.020988,-0.032886,...,0.035231,-0.000288,0.062254,0.007914,0.032241,-0.018615,-0.059511,0.072126,-0.037370,Advanced LLM Prompting
2,-0.028630,0.014242,-0.084164,0.006628,0.054243,0.067711,0.027371,-0.006613,-0.034239,-0.014033,...,0.020441,0.016043,0.042599,0.010227,0.028914,-0.026917,-0.046186,0.094621,-0.040274,Advanced LLM Prompting
3,-0.005000,0.003826,-0.077749,-0.017940,0.028686,0.041942,0.004677,0.013499,-0.024628,-0.017825,...,0.016375,0.028125,0.043700,-0.006213,0.003327,-0.021241,0.000565,0.114473,-0.067262,Advanced LLM Prompting
4,-0.030237,-0.001334,-0.048906,0.036134,0.053802,0.039611,0.050056,-0.000391,-0.026402,-0.024241,...,0.052654,-0.022873,0.049713,0.004830,-0.002995,-0.079745,-0.064036,0.085545,-0.041079,Advanced LLM Prompting
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2076,-0.030157,-0.028569,-0.021154,0.016188,0.068491,0.023981,0.015260,0.004613,-0.036075,0.004965,...,0.046838,-0.021308,0.019422,-0.000578,0.012434,-0.052346,-0.039256,0.068077,-0.039344,Tool Use for LLMs
2077,-0.014586,-0.026005,-0.034889,0.012624,0.038500,0.025602,0.022767,-0.018008,-0.014859,0.007430,...,0.025851,0.010793,0.071052,-0.011569,0.007202,-0.047317,-0.035459,0.110827,0.016157,Tool Use for LLMs
2078,0.015566,-0.020361,-0.061145,0.008080,0.042109,0.039548,0.030638,0.016730,0.003772,-0.008397,...,0.016610,-0.015382,0.032689,-0.003462,0.031347,-0.042191,-0.074996,0.068875,-0.042835,Tool Use for LLMs
2079,-0.037808,-0.005392,0.003561,0.045136,0.080824,0.048209,0.039533,-0.003626,0.007740,-0.012173,...,0.065782,-0.037942,0.039202,0.002256,0.010805,-0.026725,-0.074898,0.088672,-0.045846,Tool Use for LLMs


In [ ]:
# from sklearn.decomposition import PCA
# pca = PCA(n_components=700)
# # data = data.drop(columns=['file'])
# label = 'label'
# dx = pca.fit_transform(data.drop(columns=[label]))
# lc = data[label]
# data = pd.DataFrame(dx)
# data['pattern'] = lc

### NN Architecture

In [ ]:
TARGET_COLUMN = "pattern"

ARTIFACT_DIR = Path("../models/pattern_nn_classifier").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "pattern_classifier.keras"
SCALER_PATH = ARTIFACT_DIR / "scaler.joblib"
ENCODER_PATH = ARTIFACT_DIR / "label_encoder.joblib"
METADATA_PATH = ARTIFACT_DIR / "metadata.json"

numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_features:
    raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

X = data[numeric_features].fillna(0.0).values
y = data[TARGET_COLUMN].astype(str).values
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

X_train, X_temp, y_train_enc, y_temp_enc = train_test_split(
    X,
    y_encoded,
    test_size=0.25,
    random_state=42,
    stratify=y_encoded,
)
X_val, X_test, y_val_enc, y_test_enc = train_test_split(
    X_temp,
    y_temp_enc,
    test_size=0.5,
    random_state=42,
    stratify=y_temp_enc,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

y_train = tf.keras.utils.to_categorical(y_train_enc, num_classes)
y_val = tf.keras.utils.to_categorical(y_val_enc, num_classes)
y_test = tf.keras.utils.to_categorical(y_test_enc, num_classes)

def build_classifier(input_dim: int, num_classes: int) -> Sequential:
    """Return a tuned dense network regularized for high-dimensional embeddings."""
    regularizer = tf.keras.regularizers.l2(1e-4)
    return Sequential(
        [
            tf.keras.Input(shape=(input_dim,)),
            Dense(768, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.35),
            Dense(512, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.3),
            Dense(256, activation="relu", kernel_regularizer=regularizer),
            BatchNormalization(),
            Dropout(0.25),
            Dense(128, activation="relu"),
            Dropout(0.2),
            Dense(num_classes, activation="softmax"),
        ]
    )

model = build_classifier(X_train.shape[1], num_classes)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc")],
)

callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=20, min_delta=1e-4, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=8, min_lr=1e-5),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=768,
    callbacks=callbacks,
    verbose=1,
)

test_loss, test_acc, test_top3 = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f} | Test top-3 accuracy: {test_top3:.4f}")

y_pred = model.predict(X_test)
y_pred_labels = y_pred.argmax(axis=1)
report = classification_report(
    y_test_enc,
    y_pred_labels,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
class_breakdown = (
    report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
)

print("\nKey metrics:")
display(summary)
print("\nTop classes by support:")
display(class_breakdown)

# Persist artifacts for downstream inference pipelines
model.save(MODEL_PATH, include_optimizer=True)
joblib.dump(scaler, SCALER_PATH)
joblib.dump(label_encoder, ENCODER_PATH)
metadata = {
    "target_column": TARGET_COLUMN,
    "numeric_features": numeric_features,
    "num_classes": num_classes,
    "label_classes": label_encoder.classes_.tolist(),
    "test_metrics": {
        "loss": float(test_loss),
        "accuracy": float(test_acc),
        "top3_accuracy": float(test_top3),
    },
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2))
print(f"Saved model to {MODEL_PATH}")
print(f"Saved scaler to {SCALER_PATH}")
print(f"Saved label encoder to {ENCODER_PATH}")
print(f"Saved metadata to {METADATA_PATH}")


Epoch 1/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.3064 - loss: 2.7199 - top3_acc: 0.4814 - val_accuracy: 0.4808 - val_loss: 1.9820 - val_top3_acc: 0.6769 - learning_rate: 0.0010
Epoch 2/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6064 - loss: 1.5307 - top3_acc: 0.7917 - val_accuracy: 0.5500 - val_loss: 1.5616 - val_top3_acc: 0.7885 - learning_rate: 0.0010
Epoch 3/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7295 - loss: 1.1169 - top3_acc: 0.8853 - val_accuracy: 0.6462 - val_loss: 1.3073 - val_top3_acc: 0.8615 - learning_rate: 0.0010
Epoch 4/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7853 - loss: 0.8995 - top3_acc: 0.9250 - val_accuracy: 0.6615 - val_loss: 1.2239 - val_top3_acc: 0.8846 - learning_rate: 0.0010
Epoch 5/200
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8346 - loss: 0.7608 - top3_acc: 0.9500 - val_accuracy: 0.6769 - val_loss: 1.1436 - val_top3_acc: 0.9038 - learning_rate: 0.0010
Epoch 6/200
25/25 ━━━━━━━━━━━━

,precision,recall,f1-score,support
accuracy,0.766,0.766,0.766,0.766
macro avg,0.778,0.761,0.753,261.000
weighted avg,0.793,0.766,0.765,261.000



Top classes by support:


,precision,recall,f1-score,support
Classical Models,0.900,0.692,0.783,13.0
LLM Results Evaluation,0.625,0.769,0.690,13.0
Model Abstraction Pattern,1.000,0.923,0.960,13.0
Preprocessing Text and Numerical Data,0.929,1.000,0.963,13.0
Cross-lingual LLM Prompting,1.000,1.000,1.000,12.0
Advanced LLM Prompting,0.667,0.667,0.667,12.0
Integrating External Knlowladge with LLM,0.833,0.833,0.833,12.0
Explainable AI (XAI) Techniques,0.917,0.917,0.917,12.0


Saved model to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/pattern_classifier.keras
Saved scaler to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/scaler.joblib
Saved label encoder to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/label_encoder.joblib
Saved metadata to /home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_nn_classifier/metadata.json


In [44]:
def load_pattern_classifier(
    model_path: Path = MODEL_PATH,
    scaler_path: Path = SCALER_PATH,
    encoder_path: Path = ENCODER_PATH,
):
    """Load the saved NN, scaler, and label encoder for inference."""
    model = tf.keras.models.load_model(model_path)
    scaler = joblib.load(scaler_path)
    encoder = joblib.load(encoder_path)
    return model, scaler, encoder


def predict_pattern_probabilities(
    embeddings: np.ndarray,
    model: tf.keras.Model,
    scaler: StandardScaler,
    encoder: LabelEncoder,
) -> pd.DataFrame:
    """Return class probability dataframe for the provided embeddings."""
    embeddings = np.atleast_2d(embeddings)
    scaled = scaler.transform(embeddings)
    probs = model.predict(scaled)
    return pd.DataFrame(probs, columns=encoder.classes_)


# Example usage (uncomment to run):
loaded_model, loaded_scaler, loaded_encoder = load_pattern_classifier()
sample_probs = predict_pattern_probabilities(X_test[:5], loaded_model, loaded_scaler, loaded_encoder)
display(sample_probs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step


,Advanced LLM Prompting,Classical Models,Cross-lingual LLM Prompting,Enhanced User Intent Comprehension with LLMs,Explainable AI (XAI) Techniques,Integrating External Knlowladge with LLM,LLM Agent Training & Alignment,LLM Code Execution for Precision,LLM Context Management,LLM KV Cache Optimization,...,"LLM based Planning, Iterative Optimizations and ReAct, or Reasoning, Think Step by step, XoT",LLMs for Recommender Systems,Model Abstraction Pattern,Modular LLM Agent Architectures,Preprocessing Text and Numerical Data,"Reliable, Transparent, & Augmented LLMs",Retrieval Augmented Generation(RAG) Optimization for LLMs,Structured Output & Formatting for LLMs,Tool Use for LLMs,nan
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,4.420330e-21,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0


### Logistic Regression classifier

In [52]:
# Logistic regression stage intentionally skipped per latest workflow requirements.
# The end-to-end classifier now relies solely on the neural network above.
numeric_features = data.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_features:
    raise ValueError("No numeric features detected – please ensure embeddings/features are present.")

X = data[numeric_features].fillna(0.0).values
y = data[TARGET_COLUMN].astype(str).values
scaler = StandardScaler()
X = scaler.fit_transform(X)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.25, random_state=42,stratify=y_encoded
)

In [54]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000,n_jobs=-1)
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)
report = classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
class_breakdown = (
    report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
)
summary

,precision,recall,f1-score,support
accuracy,0.745,0.745,0.745,0.745
macro avg,0.744,0.736,0.736,521.000
weighted avg,0.755,0.745,0.745,521.000


In [56]:
# Save model
ARTIFACT_DIR = Path("../models/pattern_logreg_classifier").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(logreg, ARTIFACT_DIR / "logistic_regression_model.joblib")
joblib.dump(label_encoder, ARTIFACT_DIR / "label_encoder.joblib")
joblib.dump(scaler, ARTIFACT_DIR / "scaler.joblib")

['/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_logreg_classifier/scaler.joblib']

### SVC

In [55]:
#import svc from sklearn.svm import SVC
from sklearn.svm import SVC
svc = SVC(probability=True)
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)
report = classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
summary = report_df.loc[["accuracy", "macro avg", "weighted avg"]].round(3)
class_breakdown = (
    report_df.drop(index=["accuracy", "macro avg", "weighted avg"]).sort_values("support", ascending=False).head(8).round(3)
)   
summary

,precision,recall,f1-score,support
accuracy,0.741,0.741,0.741,0.741
macro avg,0.744,0.734,0.732,521.000
weighted avg,0.756,0.741,0.742,521.000


In [51]:
ARTIFACT_DIR = Path("../models/pattern_svc_classifier").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "svc_model.joblib"
joblib.dump(svc, MODEL_PATH)
joblib.dump(label_encoder, ARTIFACT_DIR / "label_encoder.joblib")

['/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/models/pattern_svc_classifier/label_encoder.joblib']